# MegNIST paper reproduction

This Colab notebook reproduces the **computational analysis figures (Figures 5–8)** in the MegNIST *Scientific Data* manuscript from the public release.

- **Figure 5:** model selection and generalisation
- **Figure 6:** 100,000-shuffle label-permutation test
- **Figure 7:** temporal-window channel-permutation importance
- **Figure 8:** baseline-corrected planar-gradiometer RMS topographies

Figures 1–4 are experimental/data-organisation schematics rather than outputs of the validation analysis, so they are not regenerated here.

### Reproducibility choices

- Data and released analysis artefacts come from the public Hugging Face dataset `pnpl/MegNIST`.
- No Hugging Face token or login is required.
- The representative model is the released **1024-unit, seed-47** MLP.
- Standardisation statistics are computed **only from the training split** and then applied unchanged to validation/test.
- The standardisation calculation reproduces the Welford procedure used for the released model.
- Stochastic post-hoc analyses use **analysis seed 42**.
- Figure 7 uses the **actual HDF5 `times` vector**, not an approximate generated time axis.
- Figure 7 reproduces the historical manipulation: **channel rows are permuted within each trial and temporal window**. This is not conventional across-trial PFI.
- Figures are saved as PDF and 300-dpi PNG.

> **While the PNPL loader PR is open**, the install cell below uses the `megnist-dataloader` branch. After the PR is merged, replace that line with `pip install pnpl` (or install `main`).

In [ ]:
# Colab setup
# TEMPORARY while the PNPL MegNIST loader PR is open:
%pip install -q "git+https://github.com/neural-processing-lab/pnpl.git@megnist-dataloader"

%pip install -q \
    "numpy>=1.24,<3" \
    "scipy>=1.10,<2" \
    "matplotlib>=3.7,<4" \
    "h5py>=3.10,<4" \
    "mne>=1.4.2,<2" \
    "huggingface-hub>=0.19" \
    "tqdm>=4.66"

In [ ]:
from pathlib import Path
import gc
import math
import pickle
import shutil
import time

import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import mne
import numpy as np
from scipy import stats
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

HF_REPO = "pnpl/MegNIST"

WORK_DIR = Path("/content/MegNIST_paper")
DATA_DIR = WORK_DIR / "data"
FIG_DIR = WORK_DIR / "figures"
CACHE_DIR = WORK_DIR / "cache"
for p in (DATA_DIR, FIG_DIR, CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SEED = 47
ANALYSIS_SEED = 42
N_PERMUTATIONS = 100_000
PFI_REPEATS = 10
PFI_WINDOW_SAMPLES = 25
PFI_ALPHA = 0.05

print("Device:", DEVICE)
print("Work directory:", WORK_DIR)

## 1. PNPL loader smoke test

This first uses the new public `MegNIST` loader exactly as a user would. We keep `standardize=False` because the paper uses its own **training-fitted feature-wise standardisation** rather than PNPL's optional channel-wise normalisation.

In [ ]:
from pnpl.datasets import MegNIST

smoke = MegNIST(
    data_path=str(DATA_DIR),
    partition="validation",
    standardize=False,
)
x0, y0 = smoke[0]

print("validation samples:", len(smoke))
print("sample shape:", tuple(x0.shape))
print("first label:", int(y0))
print("sampling frequency:", smoke.sfreq)

assert len(smoke) == 1000
assert tuple(x0.shape) == (306, 250)
assert smoke.sfreq == 250.0

del smoke, x0, y0
gc.collect()

## 2. Download the released paper artefacts

The notebook uses the released train/validation/test HDF5 files, the representative seed-47 checkpoint, and the stored multi-seed architecture-search results. All are public.

In [ ]:
REMOTE_FILES = {
    "train": "derivatives/serialised/train.h5",
    "val": "derivatives/serialised/val.h5",
    "test": "derivatives/serialised/test.h5",
    "model": "derivatives/models/best_model_seed47.pth",
    "results": "derivatives/results/results_all_architectures.pkl",
}

paths = {}
for key, filename in REMOTE_FILES.items():
    print(f"{key:>7}: {filename}")
    paths[key] = Path(
        hf_hub_download(
            repo_id=HF_REPO,
            repo_type="dataset",
            filename=filename,
            local_dir=DATA_DIR,
        )
    )

for split, expected_n in [("train", 10_000), ("val", 1_000), ("test", 1_000)]:
    with h5py.File(paths[split], "r") as f:
        assert f["data"].shape == (expected_n, 306, 250)
        assert f["labels"].shape == (expected_n,)
        times = np.asarray(f["times"][:], dtype=float)
        sfreq = float(round(1.0 / np.median(np.diff(times))))
        assert sfreq == 250.0
        print(
            f"{split:>5}: {f['data'].shape}, "
            f"time={times[0]*1000:.1f}..{times[-1]*1000:.1f} ms, "
            f"sfreq={sfreq:.1f} Hz"
        )

## 3. Reproduce the paper's training-set standardisation

The released representative model was trained on flattened 306 × 250 epochs using one mean and standard deviation for **each sensor–time feature**. Statistics were computed from the 10,000 training trials only and then reused unchanged for validation and test data.

The code below reproduces the original sequential **Welford** calculation. The result is cached locally so it only needs to be computed once per Colab workspace.

In [ ]:
STATS_CACHE = CACHE_DIR / "training_feature_standardisation_welford.npz"

def compute_feature_statistics_welford(h5_path, batch_size=1000):
    # Exact Welford procedure used by the released baseline training code.
    start_time = time.time()

    with h5py.File(h5_path, "r") as f:
        n_samples = len(f["data"])
        sample_shape = f["data"][0].shape
        n_features = int(np.prod(sample_shape))

        mean = np.zeros(n_features, dtype=np.float64)
        M2 = np.zeros(n_features, dtype=np.float64)
        count = 0

        starts = range(0, n_samples, batch_size)
        for start_idx in tqdm(
            starts,
            total=math.ceil(n_samples / batch_size),
            desc="Training statistics",
        ):
            end_idx = min(start_idx + batch_size, n_samples)
            batch = f["data"][start_idx:end_idx]
            batch_flat = batch.reshape(batch.shape[0], -1)

            # Deliberately sequential: matches the released training implementation.
            for row in batch_flat:
                count += 1
                delta = row - mean
                mean += delta / count
                delta2 = row - mean
                M2 += delta * delta2

        std = np.sqrt(M2 / count)
        std[std == 0] = 1.0

    print(
        f"Computed statistics from {count:,} training samples "
        f"in {time.time() - start_time:.1f} s"
    )
    return mean, std, count


if STATS_CACHE.exists():
    cached = np.load(STATS_CACHE)
    train_mean = cached["mean"]
    train_std = cached["std"]
    train_count = int(cached["count"])
    print("Loaded cached training standardisation:", STATS_CACHE)
else:
    train_mean, train_std, train_count = compute_feature_statistics_welford(
        paths["train"]
    )
    np.savez_compressed(
        STATS_CACHE,
        mean=train_mean,
        std=train_std,
        count=np.asarray(train_count),
    )
    print("Saved:", STATS_CACHE)

assert train_count == 10_000
assert train_mean.shape == (306 * 250,)
assert train_std.shape == (306 * 250,)
print("features:", len(train_mean))

In [ ]:
class PaperH5Dataset(Dataset):
    def __init__(self, h5_path, mean, std):
        self.h5_path = str(h5_path)
        self.mean = mean
        self.std = std
        with h5py.File(self.h5_path, "r") as f:
            self.length = len(f["data"])
            self.data_shape = tuple(f["data"].shape[1:])

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.h5_path, "r") as f:
            x = np.asarray(f["data"][idx]).reshape(-1)
            y = int(f["labels"][idx])

        # Same numerical path as released training:
        # float32 raw -> float64 Welford stats -> float32 tensor.
        x = ((x - self.mean) / self.std).astype(np.float32, copy=False)
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


def make_loader(h5_path, batch_size=32):
    return DataLoader(
        PaperH5Dataset(h5_path, train_mean, train_std),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

val_loader = make_loader(paths["val"])
test_loader = make_loader(paths["test"])

## 4. Load and verify the released representative model

The paper's selected architecture has one hidden layer with **1,024 units**, ReLU activation and a 10-way output. The representative model is **seed 47**.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


checkpoint = torch.load(paths["model"], map_location=DEVICE, weights_only=False)
arch = checkpoint["architecture"]

model = SimpleMLP(
    input_size=int(arch["input_size"]),
    hidden_size=int(arch["hidden_size"]),
    num_classes=int(arch["output_size"]),
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("architecture:", arch)
print("training info:", checkpoint.get("training_info", {}))


def predict_loader(model, loader, device):
    ys, preds = [], []
    model.eval()
    with torch.inference_mode():
        for x, y in tqdm(loader, leave=False):
            logits = model(x.to(device, non_blocking=True))
            pred = logits.argmax(dim=1).cpu()
            ys.append(y.cpu())
            preds.append(pred)
    return torch.cat(ys).numpy(), torch.cat(preds).numpy()


y_val, pred_val = predict_loader(model, val_loader, DEVICE)
y_test, pred_test = predict_loader(model, test_loader, DEVICE)

val_acc = 100 * np.mean(pred_val == y_val)
test_acc = 100 * np.mean(pred_test == y_test)

print(f"Validation accuracy: {val_acc:.1f}%")
print(f"Test accuracy:       {test_acc:.1f}%")

assert np.isclose(val_acc, 27.1, atol=0.05), val_acc
assert np.isclose(test_acc, 22.6, atol=0.05), test_acc

# Figure 5 — Model selection and generalisation

The figure is generated from the released 7-architecture × 10-seed search results. Plotting matches the manuscript: half-violins, white box plots, individual coloured seed points, green→yellow→orange palette, pale-blue highlighting of the selected 1,024-unit model, and a yellow star in the validation panel.

In [ ]:
with open(paths["results"], "rb") as f:
    architecture_results = pickle.load(f)

architectures = [int(v) for v in architecture_results["architectures"]]
val_data = {
    int(h): np.asarray(architecture_results["val_accuracies"][h], dtype=float)
    for h in architecture_results["val_accuracies"]
}
test_data = {
    int(h): np.asarray(architecture_results["test_accuracies"][h], dtype=float)
    for h in architecture_results["test_accuracies"]
}

selected_arch = int(architecture_results["best_architecture"])
selected_idx = architectures.index(selected_arch)

palette = [
    "#1B5E20", "#388E3C", "#66BB6A", "#AED581",
    "#DCE775", "#FFEB3B", "#FF6F00",
]

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})


def draw_rain_panel(ax, data_dict, ylabel, seed=2026):
    rng = np.random.RandomState(seed)

    for i, (h, color) in enumerate(zip(architectures, palette)):
        vals = np.asarray(data_dict[h], dtype=float)

        violin = ax.violinplot(
            vals,
            positions=[i - 0.18],
            widths=0.58,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        body = violin["bodies"][0]
        body.set_facecolor(color)
        body.set_edgecolor("#555555")
        body.set_linewidth(0.8)
        body.set_alpha(0.88)

        path = body.get_paths()[0]
        verts = path.vertices
        centre = i - 0.18
        verts[:, 0] = np.minimum(verts[:, 0], centre)

        ax.boxplot(
            vals,
            positions=[i],
            widths=0.23,
            patch_artist=True,
            showfliers=True,
            medianprops=dict(color="#222222", linewidth=1.8),
            boxprops=dict(facecolor="white", edgecolor="#555555", linewidth=1.1),
            whiskerprops=dict(color="#555555", linewidth=1.1),
            capprops=dict(color="#555555", linewidth=1.1),
            flierprops=dict(
                marker="o", markersize=4, markerfacecolor="white",
                markeredgecolor="#777777", alpha=0.9,
            ),
        )

        jitter = rng.uniform(-0.035, 0.035, size=len(vals))
        ax.scatter(
            np.full(len(vals), i + 0.18) + jitter,
            vals,
            s=28,
            color=color,
            edgecolor="none",
            alpha=0.95,
            zorder=4,
        )

    ax.axvspan(
        selected_idx - 0.52,
        selected_idx + 0.46,
        color="#DCEAF7",
        alpha=0.70,
        zorder=-2,
    )
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(len(architectures)))
    ax.set_xticklabels([str(h) for h in architectures])
    ax.grid(axis="y", alpha=0.32, linewidth=0.8)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#333333")
        spine.set_linewidth(1.0)


fig, axes = plt.subplots(
    2, 1, figsize=(10, 5.2), sharex=True,
    gridspec_kw={"hspace": 0.08},
)

draw_rain_panel(axes[0], val_data, "Validation accuracy (%)")
draw_rain_panel(axes[1], test_data, "Test accuracy (%)")

axes[0].set_ylim(23.6, 29.2)
axes[1].set_ylim(19.4, 25.0)
axes[1].set_xlabel("Hidden layer size")

axes[0].plot(
    selected_idx,
    28.75,
    marker="*",
    markersize=13,
    markerfacecolor="#FFEB3B",
    markeredgecolor="#333333",
    markeredgewidth=1.0,
    linestyle="none",
    zorder=10,
)

fig.tight_layout()
fig.savefig(
    FIG_DIR / "figure5_model_selection.pdf",
    bbox_inches="tight",
    facecolor="white",
)
fig.savefig(
    FIG_DIR / "figure5_model_selection.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

val_means = {h: float(np.mean(val_data[h])) for h in architectures}
best_arch = max(val_means, key=val_means.get)

print("Selected architecture:", best_arch)
print(
    f"1024 validation: {np.mean(val_data[1024]):.1f} ± "
    f"{np.std(val_data[1024]):.1f}%"
)
print(
    f"1024 test:       {np.mean(test_data[1024]):.1f} ± "
    f"{np.std(test_data[1024]):.1f}%"
)
assert best_arch == 1024

# Figure 6 — Decoding significance

Predictions from the released representative model are held fixed. Test labels are randomly shuffled **100,000 times** using analysis seed 42. The empirical one-sided p-value uses the standard **+1 correction**.

The notebook prints the regenerated 99th-percentile threshold. Use that computed value consistently in the final manuscript text and caption.

In [ ]:
rng_perm = np.random.RandomState(ANALYSIS_SEED)

observed_test_acc = 100 * np.mean(pred_test == y_test)
null_accs = np.empty(N_PERMUTATIONS, dtype=np.float32)

for i in tqdm(range(N_PERMUTATIONS), desc="Label permutations"):
    shuffled_labels = rng_perm.permutation(y_test)
    null_accs[i] = 100 * np.mean(pred_test == shuffled_labels)

chance_level = 10.0
alpha = 0.01
threshold_99 = float(np.percentile(null_accs, 99))
n_extreme = int(np.sum(null_accs >= observed_test_acc))
p_value = (n_extreme + 1) / (N_PERMUTATIONS + 1)

print(f"Observed accuracy:        {observed_test_acc:.2f}%")
print(f"Null median:              {np.median(null_accs):.2f}%")
print(f"Null mean ± SD:           {np.mean(null_accs):.2f} ± {np.std(null_accs):.2f}%")
print(f"99th percentile:          {threshold_99:.2f}%")
print(f"Permutations >= observed: {n_extreme}")
print(f"+1 corrected p-value:     {p_value:.8f}")

np.savez_compressed(
    CACHE_DIR / "figure6_permutation_test_seed42.npz",
    observed_test_acc=observed_test_acc,
    null_accs=null_accs,
    threshold_99=threshold_99,
    p_value=p_value,
    seed=ANALYSIS_SEED,
)

fig, ax = plt.subplots(figsize=(10, 3.5))

ax.hist(
    null_accs,
    bins=50,
    color="#C8E6C9",
    alpha=0.85,
    edgecolor="#66BB6A",
    linewidth=1.0,
    label="Null distribution",
)
ax.axvline(
    chance_level,
    color="#388E3C",
    linestyle="--",
    linewidth=2.2,
    label=f"Chance level ({chance_level:.1f}%)",
    zorder=5,
)
ax.axvline(
    threshold_99,
    color="#1976D2",
    linestyle=":",
    linewidth=2.2,
    label=rf"$\alpha$ = 0.01 threshold ({threshold_99:.1f}%)",
    zorder=5,
)
ax.axvline(
    observed_test_acc,
    color="#FFEB3B",
    linestyle="-",
    linewidth=3.5,
    label=f"Observed ({observed_test_acc:.1f}%)",
    zorder=10,
)

ax.set_xlabel("Test accuracy (%)")
ax.set_ylabel("Frequency")
ax.set_xlim(5.0, 23.5)
ax.grid(alpha=0.30, linewidth=0.8)
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.69, 0.98),
    frameon=True,
    fancybox=False,
    edgecolor="#555555",
    fontsize=9,
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#333333")
    spine.set_linewidth(1.0)

fig.tight_layout()
fig.savefig(
    FIG_DIR / "figure6_permutation_test.pdf",
    bbox_inches="tight",
    facecolor="white",
)
fig.savefig(
    FIG_DIR / "figure6_permutation_test.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

# Figure 7 — Temporal-window channel-permutation importance

This reproduces the manipulation actually used in the original analysis:

1. use the **validation** split;
2. standardise it with **training-derived feature-wise statistics**;
3. for each of the 250 temporal centres, define a 25-sample (~100 ms) window;
4. within each trial, randomly **permute channel rows** inside that window;
5. repeat 10 times and measure the accuracy drop from the unperturbed validation accuracy;
6. use one-sided one-sample t-tests and Bonferroni correction across 250 temporal centres.

This should be described as **temporal-window channel-permutation importance/perturbation**, not conventional across-trial permutation feature importance.

The result is cached after the first run. Delete the cache or set `FORCE_RECOMPUTE_FIG7=True` to regenerate it.

In [ ]:
FIG7_CACHE = (
    CACHE_DIR
    / f"figure7_temporal_importance_seed{ANALYSIS_SEED}_"
      f"win{PFI_WINDOW_SAMPLES}_rep{PFI_REPEATS}.npz"
)
FORCE_RECOMPUTE_FIG7 = False


def load_standardised_split_3d(h5_path, mean, std, batch_size=100):
    with h5py.File(h5_path, "r") as f:
        n, c, t = f["data"].shape
        y = np.asarray(f["labels"][:], dtype=np.int64)
        times = np.asarray(f["times"][:], dtype=np.float64)
        out = np.empty((n, c, t), dtype=np.float32)

        for start in tqdm(
            range(0, n, batch_size),
            desc="Standardising validation",
        ):
            stop = min(start + batch_size, n)
            raw = np.asarray(f["data"][start:stop])
            flat = raw.reshape(stop - start, -1)
            z = (flat - mean[None, :]) / std[None, :]
            out[start:stop] = z.reshape(stop - start, c, t).astype(np.float32)

    return out, y, times


def accuracy_from_3d(model, x3d, labels, device):
    x = torch.from_numpy(x3d.reshape(len(x3d), -1)).to(device)
    y = torch.from_numpy(labels).to(device)
    with torch.inference_mode():
        pred = model(x).argmax(dim=1)
    return 100 * (pred == y).float().mean().item()


def temporal_channel_permutation_importance(
    model,
    x3d,
    labels,
    times,
    device,
    n_repeats=10,
    window_size=25,
    seed=42,
):
    n_samples, n_channels, n_timepoints = x3d.shape

    baseline_acc = accuracy_from_3d(model, x3d, labels, device)
    print(f"Baseline validation accuracy: {baseline_acc:.2f}%")

    all_shuffled_accs = np.zeros(
        (n_timepoints, n_repeats),
        dtype=np.float32,
    )

    # Legacy RandomState deliberately matches np.random.seed(...) /
    # np.random.permutation semantics used by the historical notebook.
    rng = np.random.RandomState(seed)

    for t_center in tqdm(range(n_timepoints), desc="Temporal windows"):
        t_start = max(0, t_center - window_size // 2)
        t_end = min(
            n_timepoints,
            t_center + window_size // 2 + 1,
        )

        for rep in range(n_repeats):
            x_shuffled = x3d.copy()

            # Historical operation: independently permute CHANNEL ROWS
            # for each trial, but only inside the current temporal window.
            for sample_idx in range(n_samples):
                permutation = rng.permutation(n_channels)
                x_shuffled[
                    sample_idx, :, t_start:t_end
                ] = x_shuffled[
                    sample_idx, permutation, t_start:t_end
                ]

            all_shuffled_accs[t_center, rep] = accuracy_from_3d(
                model,
                x_shuffled,
                labels,
                device,
            )

    importance = baseline_acc - all_shuffled_accs.mean(axis=1)
    shuffled_std = all_shuffled_accs.std(axis=1)
    pvalues = np.empty(n_timepoints, dtype=float)

    for t in range(n_timepoints):
        drops = baseline_acc - all_shuffled_accs[t]
        _, pvalues[t] = stats.ttest_1samp(
            drops,
            0.0,
            alternative="greater",
        )

    bonferroni_threshold = PFI_ALPHA / n_timepoints
    peak_idx = int(np.argmax(importance))
    peak_time_ms = float(times[peak_idx] * 1000)
    peak_importance = float(importance[peak_idx])

    return {
        "importance_mean": importance,
        "importance_std": shuffled_std,
        "pvalues": pvalues,
        "all_shuffled_accs": all_shuffled_accs,
        "baseline_acc": np.asarray(baseline_acc),
        "bonferroni_threshold": np.asarray(bonferroni_threshold),
        "window_size": np.asarray(window_size),
        "times": times,
        "peak_idx": np.asarray(peak_idx),
        "peak_time_ms": np.asarray(peak_time_ms),
        "peak_importance": np.asarray(peak_importance),
        "seed": np.asarray(seed),
    }


if FIG7_CACHE.exists() and not FORCE_RECOMPUTE_FIG7:
    fig7 = dict(np.load(FIG7_CACHE))
    print("Loaded cached Figure 7 analysis:", FIG7_CACHE)
else:
    x_val_3d, y_val_3d, val_times = load_standardised_split_3d(
        paths["val"],
        train_mean,
        train_std,
    )

    fig7 = temporal_channel_permutation_importance(
        model,
        x_val_3d,
        y_val_3d,
        val_times,
        device=DEVICE,
        n_repeats=PFI_REPEATS,
        window_size=PFI_WINDOW_SAMPLES,
        seed=ANALYSIS_SEED,
    )
    np.savez_compressed(FIG7_CACHE, **fig7)
    print("Saved:", FIG7_CACHE)

    del x_val_3d, y_val_3d
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

importance = np.asarray(fig7["importance_mean"])
std = np.asarray(fig7["importance_std"])
pvalues = np.asarray(fig7["pvalues"])
all_shuffled_accs = np.asarray(fig7["all_shuffled_accs"])
times_ms = np.asarray(fig7["times"]) * 1000
baseline_acc_fig7 = float(np.asarray(fig7["baseline_acc"]))
bonferroni_threshold = float(
    np.asarray(fig7["bonferroni_threshold"])
)
peak_idx = int(np.asarray(fig7["peak_idx"]))
peak_time_ms = float(np.asarray(fig7["peak_time_ms"]))
peak_importance = float(np.asarray(fig7["peak_importance"]))

print(f"Figure 7 baseline accuracy: {baseline_acc_fig7:.2f}%")
print(
    f"Figure 7 peak: {peak_time_ms:.1f} ms, "
    f"{peak_importance:.2f}% drop"
)
print(
    f"Significant time points: "
    f"{np.sum(pvalues < bonferroni_threshold)}/{len(pvalues)} "
    f"(Bonferroni p < {bonferroni_threshold:.1e})"
)

In [ ]:
sem = std / np.sqrt(all_shuffled_accs.shape[1])
ci95 = 1.96 * sem
significant = pvalues < bonferroni_threshold

fig, ax = plt.subplots(figsize=(10, 4.2))

ax.plot(
    times_ms,
    importance,
    color="steelblue",
    linewidth=2.5,
    label="Mean importance",
)
ax.fill_between(
    times_ms,
    importance - ci95,
    importance + ci95,
    color="steelblue",
    alpha=0.22,
    linewidth=0,
    label="95% CI",
)

if np.any(significant):
    ax.scatter(
        times_ms[significant],
        importance[significant],
        s=27,
        color="black",
        edgecolor="none",
        zorder=6,
        label=(
            f"Significant (p<{bonferroni_threshold:.1e}, "
            f"{PFI_WINDOW_SAMPLES / 250 * 1000:.0f}ms windows)"
        ),
    )

ax.axvline(
    0.0,
    color="#888888",
    linestyle="--",
    linewidth=1.2,
    alpha=0.75,
    label="Stimulus onset",
)


def nearest_idx(ms):
    return int(np.argmin(np.abs(times_ms - ms)))


annotation_specs = [
    ("Early Visual", 100, 42, 1.55),
    ("Visual Word Form", 170, 145, 4.85),
    ("Lexical Access", peak_time_ms, peak_time_ms + 42, 5.05),
    ("Semantic", 400, 420, 2.10),
    ("Phonological\nEncoding", 580, 520, -1.05),
]

for text, target_ms, text_x, text_y in annotation_specs:
    idx = nearest_idx(target_ms)
    ax.annotate(
        text,
        xy=(times_ms[idx], importance[idx]),
        xytext=(text_x, text_y),
        textcoords="data",
        ha="center",
        va="center",
        fontsize=8.5,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="#AAAAAA",
            linewidth=0.8,
            alpha=0.95,
        ),
        arrowprops=dict(
            arrowstyle="-",
            color="#999999",
            linewidth=0.9,
        ),
    )

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Importance (% drop in accuracy)")
ax.set_xlim(times_ms[0], times_ms[-1])
ax.set_ylim(-2.1, 5.4)
ax.grid(alpha=0.28, linewidth=0.8)

ax.legend(
    loc="upper right",
    frameon=True,
    fancybox=False,
    edgecolor="#888888",
    fontsize=8.5,
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#333333")
    spine.set_linewidth(1.0)

fig.tight_layout()
fig.savefig(
    FIG_DIR / "figure7_temporal_importance.pdf",
    bbox_inches="tight",
    facecolor="white",
)
fig.savefig(
    FIG_DIR / "figure7_temporal_importance.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

print(
    "\nMANUSCRIPT CHECK — Figure 7:\n"
    f"  regenerated peak = {peak_time_ms:.1f} ms\n"
    f"  regenerated drop = {peak_importance:.2f}%\n"
    "Use these regenerated values in the final text/caption."
)

# Figure 8 — Spatial–temporal progression

Figure 8 uses the **unstandardised test epochs**:

- pre-stimulus baseline correction from the first stored sample to 0 ms;
- planar gradiometers;
- trial averaging;
- local RMS around each target latency (approximately ±25 ms);
- one common colour scale across all five topographies;
- the manuscript's pineapple sequential colour map.

The middle (`M250`) map is placed at the **regenerated Figure 7 peak** when that peak falls in the expected 200–300 ms range, so Figures 7 and 8 stay internally consistent.

In [ ]:
BAD_CHANNELS = [
    "MEG0911", "MEG0912", "MEG0913",
    "MEG0921", "MEG0922", "MEG0923", "MEG1411",
]

with h5py.File(paths["test"], "r") as f:
    topo_data = np.asarray(f["data"][:], dtype=np.float32)
    topo_times = np.asarray(f["times"][:], dtype=float)
    ch_names = [
        v.decode() if isinstance(v, (bytes, bytearray)) else str(v)
        for v in f["channel_names"][:]
    ]
    ch_types = [
        v.decode() if isinstance(v, (bytes, bytearray)) else str(v)
        for v in f["channel_types"][:]
    ]
    sensor_xyz = np.asarray(f["sensor_xyz"][:], dtype=float)

sfreq = float(round(1.0 / np.median(np.diff(topo_times))))
tmin = float(topo_times[0])

assert sfreq == 250.0
assert topo_data.shape == (1000, 306, 250)

info = mne.create_info(
    ch_names=ch_names,
    sfreq=sfreq,
    ch_types=ch_types,
)

for idx in range(len(ch_names)):
    info["chs"][idx]["loc"][:3] = sensor_xyz[idx]
    norm = np.linalg.norm(sensor_xyz[idx])
    if norm > 0:
        info["chs"][idx]["loc"][9:12] = sensor_xyz[idx] / norm

info["bads"] = [ch for ch in BAD_CHANNELS if ch in ch_names]

epochs = mne.EpochsArray(
    topo_data,
    info=info,
    tmin=tmin,
    verbose=False,
)
epochs.apply_baseline((tmin, 0.0))
evoked_grad = epochs.average().pick("grad")

if 200.0 <= peak_time_ms <= 300.0:
    lexical_time = peak_time_ms / 1000.0
else:
    lexical_time = 0.250

time_windows = {
    "Early Visual\n(M100)": 0.100,
    "Visual Word Form\n(M170)": 0.170,
    "Lexical Access\n(M250)": lexical_time,
    "Semantic\n(M400)": 0.400,
    "Phonological\nEncoding": 0.580,
}


def truncate_cmap(cmap, minval=0.3, maxval=1.0, N=256):
    colors = cmap(np.linspace(minval, maxval, N))
    return LinearSegmentedColormap.from_list(
        f"{cmap.name}_trunc",
        colors,
    )


pineapple_seq = LinearSegmentedColormap.from_list(
    "pineapple_seq",
    [
        "#1b5e20",
        "#388e3c",
        "#66bb6a",
        "#dce775",
        "#ffd54f",
        "#ffb300",
        "#ff8f00",
    ],
    N=256,
)
cmap_custom = truncate_cmap(
    pineapple_seq,
    minval=0.35,
    maxval=1.0,
)

half_window = int(0.025 * sfreq)


def local_rms(evoked, time_sec):
    time_idx = int(np.argmin(np.abs(evoked.times - time_sec)))
    start_idx = max(0, time_idx - half_window)
    end_idx = min(
        len(evoked.times),
        time_idx + half_window,
    )
    data_window = evoked.data[:, start_idx:end_idx]
    return np.sqrt(np.mean(data_window ** 2, axis=1))


topo_values = {
    label: local_rms(evoked_grad, t)
    for label, t in time_windows.items()
}
vmax_global = max(
    float(np.max(v))
    for v in topo_values.values()
)

fig, axes = plt.subplots(
    1,
    len(time_windows),
    figsize=(20, 4.5),
)

im = None
for idx, ((label, time_sec), ax) in enumerate(
    zip(time_windows.items(), axes)
):
    data_to_plot = topo_values[label]

    im, _ = mne.viz.plot_topomap(
        data_to_plot,
        evoked_grad.info,
        axes=ax,
        show=False,
        contours=6,
        cmap=cmap_custom,
        sensors=False,
        names=None,
        sphere=None,
        vlim=(0, vmax_global),
        size=4,
    )

    ax.set_title(label, fontsize=12, pad=10)
    actual_idx = int(
        np.argmin(np.abs(evoked_grad.times - time_sec))
    )
    actual_ms = int(
        round(evoked_grad.times[actual_idx] * 1000)
    )
    ax.text(
        0.5,
        -0.15,
        f"{actual_ms} ms",
        transform=ax.transAxes,
        ha="center",
        fontsize=11,
        weight="bold",
    )

divider = make_axes_locatable(axes[-1])
cax = divider.append_axes("right", size="5%", pad=0.2)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label("RMS activity (fT/cm)", fontsize=11)
cbar.ax.tick_params(labelsize=9)

plt.tight_layout(rect=[0, 0.02, 0.98, 0.94])
fig.savefig(
    FIG_DIR / "figure8_spatial_temporal_progression.pdf",
    bbox_inches="tight",
    facecolor="white",
)
fig.savefig(
    FIG_DIR / "figure8_spatial_temporal_progression.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

print("Figure 8 latencies:")
for label, time_sec in time_windows.items():
    actual_idx = int(
        np.argmin(np.abs(evoked_grad.times - time_sec))
    )
    actual_ms = evoked_grad.times[actual_idx] * 1000
    clean_label = label.replace("\n", " ")
    print(f"  {clean_label:<30} {actual_ms:6.1f} ms")

## Final checks

The notebook deliberately **regenerates**, rather than hard-codes, the two numbers that still need to be reconciled with the draft manuscript:

1. **Figure 6:** the empirical 99th-percentile null threshold;
2. **Figure 7:** the exact peak time and accuracy drop using the released HDF5 time vector and fixed analysis seed.

After the first clean Colab run, compare those printed values with the manuscript and update the text/captions if necessary.

In [ ]:
print("=" * 72)
print("MEGNIST PAPER REPRODUCTION SUMMARY")
print("=" * 72)
print(f"Figure 5 best architecture:       {best_arch}")
print(
    f"Figure 5 1024 val/test:           "
    f"{np.mean(val_data[1024]):.1f} ± {np.std(val_data[1024]):.1f}% / "
    f"{np.mean(test_data[1024]):.1f} ± {np.std(test_data[1024]):.1f}%"
)
print(
    f"Released seed-47 val/test:        "
    f"{val_acc:.1f}% / {test_acc:.1f}%"
)
print(
    f"Figure 6 observed test accuracy:  "
    f"{observed_test_acc:.1f}%"
)
print(
    f"Figure 6 99th percentile:         "
    f"{threshold_99:.1f}%"
)
print(
    f"Figure 6 corrected p-value:       "
    f"{p_value:.8f}"
)
print(
    f"Figure 7 peak:                    "
    f"{peak_time_ms:.1f} ms"
)
print(
    f"Figure 7 peak drop:               "
    f"{peak_importance:.2f}%"
)
print(f"Figures saved to:                 {FIG_DIR}")
print("=" * 72)

print("\nGenerated files:")
for p in sorted(FIG_DIR.iterdir()):
    print(" ", p.name)

archive = shutil.make_archive(
    str(WORK_DIR / "MegNIST_paper_figures"),
    "zip",
    root_dir=FIG_DIR,
)
print("\nFigure archive:", archive)